# Análisis de Puntos Negros - Madrid

# 1. Instalaciones y configuración general

In [ ]:
# Importar librerías, si alguna libreria falta hacer pip install libreria en la terminal
import os, warnings
import pandas as pd
import numpy as np
import folium
from pyproj import Transformer
from sklearn.neighbors import BallTree

warnings.filterwarnings("ignore")

# Resolucion de la rejilla
# 0.001 lat son 111m,  0.001 lon son 85m
GRID = 0.001
OUT_DIR  = "outputs"

# Umbrales para clasificar como punto negro
MIN_ACC_BAJO = 5
MIN_ACC_MEDIO = 10
MIN_ACC_ALTO = 25
MIN_ACC_CRITICO = 50

# Pesos para el indice de peligrosidad
W_ACC = 0.40
W_GRAVES = 0.30
W_MORT = 0.20
W_ALCOHOL = 0.10

# Crea el directorio de salida si no existe
os.makedirs(OUT_DIR, exist_ok=True)

# Transformador de coordenadas UTM a WGS84 que es lat/lon estándar
utm_to_wgs = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)
# Definir y filtrar las coordenadas que van a delimitar Madrid
BBOX = dict(lat_min=40.30, lat_max=40.60, lon_min=-3.85, lon_max=-3.55)

def filtrar_bbox(df, lat="lat", lon="lon"):
    return df[df[lat].between(BBOX["lat_min"],BBOX["lat_max"]) &
              df[lon].between(BBOX["lon_min"],BBOX["lon_max"])].copy()

# Normalizar una serie al rango [0, 9]
def norm(s):
    mn,mx = s.min(),s.max()
    return (s-mn)/(mx-mn+1e-9)

c:\Users\nuria\anaconda3\envs\dl_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


# 2. Carga del dataset optimizado

In [ ]:
print("Cargando accidentes...")

# Leer las columnas que vamos a usar del csv accidentes_madrid_optimizado
df_opt = pd.read_csv(
    "accidentes_madrid_optimizado.csv",
    usecols=["num_expediente","lesividad","positiva_alcohol","longitud","latitud"],
    low_memory=False
)

# Convierte las columnas de texto a número
df_opt["lat"] = pd.to_numeric(df_opt["latitud"],  errors="coerce")
df_opt["lon"] = pd.to_numeric(df_opt["longitud"], errors="coerce")
# Elimina filas sin coordenadas válidas
df_opt = df_opt.dropna(subset=["lat","lon"])
# Descarta registros fuera del área de Madrid
df_opt = filtrar_bbox(df_opt)

# Crea flag binario: 1 si la lesividad indica herido grave, mortal o fallecido
df_opt["grave"]   = df_opt["lesividad"].str.contains(r"24 horas|grave|mortal|falleci",case=False,na=False).astype(int)
# Solo accidentes mortales
df_opt["mortal"]  = df_opt["lesividad"].str.contains(r"mortal|falleci",case=False,na=False).astype(int)
# Si el conductor dio positivo en alcohol
df_opt["alcohol"] = df_opt["positiva_alcohol"].astype(str).str.upper().str.strip().eq("S").astype(int)
# Promedia coordenadas y toma el peor caso para la gravedad
df_opt_exp = df_opt.groupby("num_expediente").agg(
    lat=("lat","mean"), lon=("lon","mean"),
    grave=("grave","max"), mortal=("mortal","max"), alcohol=("alcohol","max")
).reset_index()
print(f"  Optimizado:   {len(df_opt_exp):>7,} accidentes unicos")

Cargando accidentes...
  Optimizado:   135,239 accidentes unicos


# 3. Carga del dataset detallado (2026)

In [ ]:
# Convertir coordenadas UTM -> WGS84
df_det = pd.read_csv(
    "300228-34-accidentes-trafico-detalle.csv",
    sep=";", encoding="utf-8-sig", low_memory=False
)

# Conviertir las coordenadas UTM de texto a float
df_det["coordenada_x_utm"] = pd.to_numeric(df_det["coordenada_x_utm"],errors="coerce")
df_det["coordenada_y_utm"] = pd.to_numeric(df_det["coordenada_y_utm"],errors="coerce")
df_det = df_det.dropna(subset=["coordenada_x_utm","coordenada_y_utm"])
lon_arr,lat_arr = utm_to_wgs.transform(df_det["coordenada_x_utm"].values,
                                        df_det["coordenada_y_utm"].values)

# Asignar las coordenadas convertidas como nuevas columnas
df_det["lat"] = lat_arr
df_det["lon"] = lon_arr

# Mismo proceso pero con el dataset optimizado
df_det = filtrar_bbox(df_det)
df_det["grave"]   = df_det["lesividad"].str.contains(r"24 horas|grave|mortal|falleci",case=False,na=False).astype(int)
df_det["mortal"]  = df_det["lesividad"].str.contains(r"mortal|falleci",case=False,na=False).astype(int)
df_det["alcohol"] = df_det["positiva_alcohol"].astype(str).str.upper().str.strip().eq("S").astype(int)
df_det_exp = df_det.groupby("num_expediente").agg(
    lat=("lat","mean"), lon=("lon","mean"),
    grave=("grave","max"), mortal=("mortal","max"), alcohol=("alcohol","max")
).reset_index()
print(f"  Detallado:    {len(df_det_exp):>7,} accidentes unicos")

  Detallado:      5,633 accidentes unicos


# 4. Fusión de ambos datasets

In [ ]:
# Unir ambos datasets sin duplicados
cols = ["num_expediente","lat","lon","grave","mortal","alcohol"]
# Apilar ambos daraframes y quitar los duplicados
df_acc = pd.concat([df_opt_exp[cols], df_det_exp[cols]], ignore_index=True)
df_acc = df_acc.drop_duplicates("num_expediente").reset_index(drop=True)
print(f"TOTAL:  {len(df_acc):>7,} accidentes con coordenadas")

TOTAL:  140,872 accidentes con coordenadas


# 5. Construcción de la rejilla de accidentalidad

Se divide la ciudad de Madrid en celdas de 90x90m cada una, cada accidente se asigna a la celda más cercana

In [ ]:
print("\nConstruyendo rejilla de accidentalidad...")

# Redondear la coordenada al entero mas cercano
df_acc["lat_g"] = (df_acc["lat"] / GRID).round() * GRID
df_acc["lon_g"] = (df_acc["lon"] / GRID).round() * GRID

# Agregar todos los accidentes que cayeron en la misma celda
grid = df_acc.groupby(["lat_g","lon_g"]).agg(
    n_accidentes=("lat", "count"),
    n_graves =("grave", "sum"),
    n_mortales =("mortal", "sum"),
    n_alcohol =("alcohol", "sum"),
    lat_centro =("lat", "mean"),
    lon_centro =("lon", "mean"),
).reset_index()

# Solo celdas con al menos el umbral minimo
pn = grid[grid["n_accidentes"] >= MIN_ACC_BAJO].copy()
print(f"  Celdas 90m con >= {MIN_ACC_BAJO} accidentes: {len(pn):,}")
print(f"  Con >= {MIN_ACC_MEDIO}: {(pn.n_accidentes>=MIN_ACC_MEDIO).sum():,}")
print(f"  Con >= {MIN_ACC_ALTO}:  {(pn.n_accidentes>=MIN_ACC_ALTO).sum():,}")
print(f"  Con >= 1 mortal:   {(pn.n_mortales>=1).sum():,}")


Construyendo rejilla de accidentalidad...
  Celdas 90m con >= 5 accidentes: 7,763
  Con >= 10: 4,014
  Con >= 25:  1,143
  Con >= 1 mortal:   152


# 6. Índice de peligrosidad y categorización

Se calcula un índice compuesto ponderado (0-1) y se asigna una categoría a cada celda

In [ ]:
# Juntar las 4 métricas en un unico score entre 0 y 1
pn["idx_peligrosidad"] = (
    W_ACC * norm(pn["n_accidentes"]) +
    W_GRAVES * norm(pn["n_graves"]) +
    W_MORT * norm(pn["n_mortales"]) +
    W_ALCOHOL* norm(pn["n_alcohol"])
).round(4)

# CLasificar las celdas por indice de peligrosidad
def categoria(row):
    if row["n_mortales"]>=1 or row["n_accidentes"]>=MIN_ACC_CRITICO:
        return "CRITICO"
    elif row["n_accidentes"]>=MIN_ACC_ALTO:
        return "ALTO"
    elif row["n_accidentes"]>=MIN_ACC_MEDIO:
        return "MEDIO"
    else:
        return "BAJO"

#Aplicar la clasificacion fila a fila y ordenar de mayor a menor
pn["categoria"] = pn.apply(categoria,axis=1)
pn = pn.sort_values("idx_peligrosidad",ascending=False).reset_index(drop=True)
pn["node_id"] = ["PN_"+str(i).zfill(4) for i in range(len(pn))]

# 7. Sensores IMD (Intensidad Media Diaria)

Para cada punto negro se cuenta cuántos sensores de tráfico hay en un radio de 500m usando BallTree.

In [ ]:
print("\n Sensores IMD...")
imd_rows = []

# Cargar dos ficheros IMD con distinto encoding
for fname,enc,sep in [
    ("202468-2-intensidad-trafico-csv.csv",  "latin-1",";"),
    ("202468-29-intensidad-trafico-csv.csv", "utf-8",  ";"),
]:
    try:
        df_i = pd.read_csv(fname,encoding=enc,sep=sep,low_memory=False)
        df_i["lat"] = pd.to_numeric(df_i["latitud"],  errors="coerce")
        df_i["lon"] = pd.to_numeric(df_i["longitud"], errors="coerce")
        df_i = filtrar_bbox(df_i.dropna(subset=["lat","lon"]))
        imd_rows.append(df_i[["lat","lon"]])
        print(f"  {fname}: {len(df_i):,} sensores")
    except Exception as e:
        print(f"  {fname}: AVISO - {e}")

# Unir ambos sensores, eliminar duplicados
if imd_rows:
    df_imd = pd.concat(imd_rows,ignore_index=True).drop_duplicates()
    tree   = BallTree(np.radians(df_imd[["lat","lon"]].values),metric="haversine") # estructura de búsqueda
    pn_rad = np.radians(pn[["lat_centro","lon_centro"]].values) # convierte los centros de los puntos a radianes
    # Contar cuantos sensores hay en 0.5 km (convertimos 0.5 km a radianes)
    pn["sensores_imd_500m"] = tree.query_radius(pn_rad, r=0.5/6371.0, count_only=True)
    print(f"  Total sensores IMD: {len(df_imd):,}")
else:
    pn["sensores_imd_500m"] = 0


 Sensores IMD...
  202468-2-intensidad-trafico-csv.csv: 5,024 sensores
  202468-29-intensidad-trafico-csv.csv: 4,370 sensores
  Total sensores IMD: 9,134


# 8. Exportar los nodos a un csv

In [ ]:
# Conjunto de columnas para el fichero final
cols_out = ["node_id","lat_centro","lon_centro","n_accidentes","n_graves",
            "n_mortales","n_alcohol","sensores_imd_500m","idx_peligrosidad","categoria"]
out_csv = os.path.join(OUT_DIR,"puntos_negros_nodos.csv")
pn[cols_out].to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n CSV nodos -> {out_csv}  ({len(pn):,} nodos)")


 CSV nodos -> outputs\puntos_negros_nodos.csv  (7,763 nodos)


# 9. Mapa interactivo

Se genera un mapa con Folium en html.

In [ ]:
# Generar mapa html
print("Generando mapa...")
# Paleta de colores
COLOR_MAP = {"CRITICO":"#c0392b","ALTO":"#e67e22","MEDIO":"#f1c40f","BAJO":"#27ae60"}

# Mapa base centrado en Madrid
m = folium.Map(location=[40.4168,-3.7038], zoom_start=12, tiles="CartoDB positron")

fg_pn = folium.FeatureGroup(name=f"Puntos Negros ({len(pn):,})")
for _,row in pn.iterrows():
    color  = COLOR_MAP.get(row["categoria"],"#95a5a6")
    radius = max(3, 2 + row["idx_peligrosidad"] * 14)
    # Contenido del html
    popup  = (f"<b>{row['node_id']}</b><br>"
              f"Categoria: <b style='color:{color}'>{row['categoria']}</b><br>"
              f"Accidentes: <b>{int(row['n_accidentes'])}</b><br>"
              f"Graves: {int(row['n_graves'])} | Mortales: {int(row['n_mortales'])}<br>"
              f"Alcohol+: {int(row['n_alcohol'])}<br>"
              f"Sensores IMD (500m): {int(row['sensores_imd_500m'])}<br>"
              f"Indice peligrosidad: <b>{row['idx_peligrosidad']:.4f}</b><br>"
              f"Lat: {row['lat_centro']:.5f} | Lon: {row['lon_centro']:.5f}")
    folium.CircleMarker(
        location=[row["lat_centro"],row["lon_centro"]],
        radius=radius,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        tooltip=f"{row['node_id']} | {row['categoria']} | {int(row['n_accidentes'])} acc.", # Al pasar el ratón
        popup=folium.Popup(popup, max_width=270), # Al hacer click
    ).add_to(fg_pn)
fg_pn.add_to(m)

# Radares existentes
try:
    df_rad = pd.read_csv("300049-1-radares-fijos-moviles-csv.csv",
                         encoding="utf-8-sig", sep=";", low_memory=False)
    df_rad.columns = df_rad.columns.str.strip()
    df_rad = df_rad.dropna(subset=["Latitud","Longitud"])
    fg_rad = folium.FeatureGroup(name="Radares existentes", show=True)
    for _,r in df_rad.iterrows():
        folium.Marker(
            location=[float(r["Latitud"]),float(r["Longitud"])],
            icon=folium.Icon(color="blue",icon="camera",prefix="fa"), # Icono de la camara
            tooltip=f"Radar | {r.get('Ubicacion','')} | {r.get('Velocidad limite','')} km/h",
        ).add_to(fg_rad)
    fg_rad.add_to(m)
except Exception as e:
    print(f"  Radares: AVISO - {e}")

# Cruces semaforizados
try:
    df_cru = pd.read_csv("300275-0-cruces-semaforizados-csv.csv",
                         encoding="latin-1", sep=";", low_memory=False)
    df_cru = df_cru.dropna(subset=["latitud","longitud"])

    # Capa desactivada por defecto para no saturar el mapa inicial
    fg_cru = folium.FeatureGroup(name="Cruces semaforizados", show=False)
    for _,r in df_cru.iterrows():
        folium.CircleMarker(
            location=[float(r["latitud"]),float(r["longitud"])],
            radius=3, color="#8e44ad", fill=True, fill_opacity=0.4, # Los puntos son morados y más transparentes
            tooltip=str(r.get("descripcion","")),
        ).add_to(fg_cru)
    fg_cru.add_to(m)
    print(f"  Cruces semaforizados: {len(df_cru)}")
except Exception as e:
    print(f"  Cruces: AVISO - {e}")

# Leyenda
m.get_root().html.add_child(folium.Element(f"""
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;
     padding:14px 18px;border-radius:10px;
     box-shadow:2px 2px 10px rgba(0,0,0,0.25);font-size:13px;line-height:2.0">
  <b>Puntos Negros — Madrid</b><br>
  <span style="color:#c0392b;font-size:16px">&#9679;</span>
    CRITICO (&ge;{MIN_ACC_CRITICO} acc. o &ge;1 mortal)<br>
  <span style="color:#e67e22;font-size:16px">&#9679;</span>
    ALTO ({MIN_ACC_ALTO}–{MIN_ACC_CRITICO-1} accidentes)<br>
  <span style="color:#f1c40f;font-size:16px">&#9679;</span>
    MEDIO ({MIN_ACC_MEDIO}–{MIN_ACC_ALTO-1} accidentes)<br>
  <span style="color:#27ae60;font-size:16px">&#9679;</span>
    BAJO ({MIN_ACC_BAJO}–{MIN_ACC_MEDIO-1} accidentes)<br>
  <small style="color:#888">Celda ~90m | Radio proporcional al indice</small>
</div>
"""))
# Control de capas para activar o desactivarlas desde el mapa
folium.LayerControl().add_to(m)

# Guardar el mapa completp como fichero HTML
out_map = os.path.join(OUT_DIR,"puntos_negros_mapa.html")
m.save(out_map)
print(f"Mapa -> {out_map}")

Generando mapa...
  Cruces semaforizados: 2503
Mapa -> outputs\puntos_negros_mapa.html


# 10. Resumen final en la terminal

In [ ]:
# Resumen final
print("RESUMEN FINAL - PUNTOS NEGROS MADRID")
for cat in ["CRITICO","ALTO","MEDIO","BAJO"]:
    n = (pn["categoria"]==cat).sum()
    print(f"    {cat:<10} {n:>5}")

print(f"\n  Archivos generados:")
print(f"  -> {out_csv}")
print(f"  -> {out_map}")

RESUMEN FINAL - PUNTOS NEGROS MADRID
  Total nodos del grafo: 7,763
    CRITICO      439
    ALTO         798
    MEDIO       2813
    BAJO        3713

  Archivos generados:
  -> outputs\puntos_negros_nodos.csv
  -> outputs\puntos_negros_mapa.html
